# 📰Pydantic 파이프라인 + 뉴스 모니터링

```
Phase . Core 학습 — Pydantic + RunnableParallel 핵심 부품
Project.   기사 → 분석 + 요약 + 태깅 + 대응 라우팅
```

## Phase 0. 환경 설정

In [ ]:
#!uv add  langchain langchain-openai python-dotenv pydantic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.6/98.6 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 548.1/548.1 kB 13.6 MB/s eta 0:00:00


In [ ]:
import os
from typing import Literal, Optional
from pydantic import BaseModel, Field
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import PydanticOutputParser, StrOutputParser
from langchain_core.runnables import RunnableParallel, RunnableLambda, RunnablePassthrough
from dotenv import load_dotenv

load_dotenv()

llm = ChatOpenAI(
    base_url="https://integrate.api.nvidia.com/v1",
    api_key=os.getenv("nvidiaapi_key"),
    model="openai/gpt-oss-20b",
    temperature=0,
)

## Phase 1. PydanticOutputParser — 자연어를 객체로

5단계 패턴: 스키마 → 파서 → 프롬프트 → LLM → 체인

In [ ]:
class Summary(BaseModel):
    title: str = Field(description="짧은 제목")
    sentiment: Literal["positive", "negative", "neutral"]
    key_points: list[str] = Field(description="핵심 2-3개")

parser = PydanticOutputParser(pydantic_object=Summary)

prompt = ChatPromptTemplate.from_messages([
    ("system",
     "텍스트를 분석합니다. 반드시 한국어로만 답변하세요. key_points 배열 안의 단어를 포함한 모든 필드 값을 "
     "예외 없이 한국어로만 작성하세요. 다른 언어(중국어, 영어 등)는 단 한 글자도 섞지 마세요. "
     "설명이나 스키마 설명 없이, 아래 예시와 똑같은 형태의 JSON 객체 하나만 실제 값으로 채워서 출력하세요.\n\n"
     "예시: {{\"title\": \"신제품 지연\", \"sentiment\": \"neutral\", \"key_points\": [\"출시일 변경\", \"고객 안내 필요\"]}}\n\n"
     "{format_instructions}"),
    ("human", "{text}"),
]).partial(format_instructions=parser.get_format_instructions())

chain_summary = prompt | llm | parser

r = chain_summary.invoke({"text": "신제품 출시를 6월에서 7월로 연기하기로 했습니다."})
print(r)

## Phase 2. RunnableParallel — 여러 체인 결합

In [ ]:
# 자연어 답장 체인 추가 (StrOutputParser)
chain_notice = (
    ChatPromptTemplate.from_messages([
        ("system",
         "팀원에게 알릴 한 줄 안내문을 딱 하나만 작성하세요. 여러 버전을 나열하지 마세요. "
         "반드시 한국어로만 작성하고 다른 언어는 섞지 마세요."),
        ("human", "{text}"),
    ]) | llm | StrOutputParser()
)

combined = RunnableParallel(summary=chain_summary, notice=chain_notice)

r = combined.invoke({"text": "신제품 출시를 6월에서 7월로 연기하기로 했습니다."})
print(r["summary"])
print(r["notice"])

## Phase 3. RunnableLambda — 함수도 체인에 (결과는 Pydantic으로)

In [7]:
class Decision(BaseModel):
    urgent: bool
    note: str

def route(x) -> Decision:
    s = x["summary"]
    return Decision(urgent=(s.sentiment == "negative"), note=s.title)

chain_routed = combined | RunnableLambda(route)

d = chain_routed.invoke({"text": "신제품 출시 연기"})
print(d)

urgent=True note='신제품 출시 연기'


## Phase 4. RunnablePassthrough — 원본 보존

In [8]:
full = RunnableParallel(
    summary=chain_summary,
    notice=chain_notice,
    original=RunnablePassthrough(),
)
r = full.invoke({"text": "신제품 출시 연기"})
print(r["original"])  # 입력 그대로 통과

{'text': '신제품 출시 연기'}


---
# 📰 Project. 뉴스 모니터링 + PR 대응 시스템

```
기사 → 분석 + 임원 요약 + 태깅 (동시)
     → 대응 라우팅 (즉시 알림 / 일일 브리핑 / 보관)
```

**모니터링 대상**: NovaTech (가상 AI 스타트업, B2B 챗봇 솔루션)

In [ ]:
SAMPLE = """[테크크런치] NovaTech, B2B 챗봇 시장 점유율 1위 등극.
LLM 챗봇을 금융·제조 대기업 50여 곳에 공급하며 전년 대비 200% 성장.
다만 일부 고객사는 응답 속도 개선이 필요하다고 지적."""

# 분석 체인 — 자사 관점에서
class NewsAnalysis(BaseModel):
    headline: str = Field(description="15자 이내 헤드라인")
    sentiment: Literal["positive", "negative", "mixed", "neutral"]
    impact_score: int = Field(ge=0, le=10, description="자사 영향도 0-10")
    contains_risk: bool = Field(description="평판/사업 리스크 언급 여부")

p1 = PydanticOutputParser(pydantic_object=NewsAnalysis)
chain_analysis = (
    ChatPromptTemplate.from_messages([
        ("system",
         "자사(NovaTech, B2B AI 챗봇 스타트업) 관점에서 기사를 분석합니다. "
         "반드시 한국어로만 답변하세요. 다른 언어(중국어, 영어 등)는 단 한 글자도 섞지 마세요. "
         "설명이나 스키마 설명 없이, 아래 예시와 똑같은 형태의 JSON 객체 하나만 실제 값으로 채워서 출력하세요.\n\n"
         "예시: {{\"headline\": \"경쟁사 신제품 출시\", \"sentiment\": \"negative\", \"impact_score\": 7, \"contains_risk\": true}}\n\n"
         "{format_instructions}"),
        ("human", "{article}"),
    ]).partial(format_instructions=p1.get_format_instructions())
    | llm | p1
)
print(chain_analysis.invoke({"article": SAMPLE}))

In [ ]:
# 임원 보고용 3문장 요약 (자연어)
chain_brief = (
    ChatPromptTemplate.from_messages([
        ("system",
         "임원 보고용 *정확히 3문장* 요약을 딱 하나만 작성하세요: 사실 / 자사 관련성 / 시사점. "
         "여러 버전을 나열하지 마세요. 반드시 한국어로만 작성하고 다른 언어는 섞지 마세요."),
        ("human", "{article}"),
    ]) | llm | StrOutputParser()
)
print(chain_brief.invoke({"article": SAMPLE}))

In [ ]:
# 태깅 체인 — 다중 선택 (list[Literal])
class NewsTags(BaseModel):
    topics: list[Literal["제품", "재무", "인사", "기술", "규제", "경쟁사", "기타"]]
    teams: list[Literal["PR", "마케팅", "프로덕트", "법무", "재무"]] = Field(
        description="이 기사를 봐야 할 사내 팀"
    )

p2 = PydanticOutputParser(pydantic_object=NewsTags)
chain_tags = (
    ChatPromptTemplate.from_messages([
        ("system",
         "기사에 적절한 주제와 관련 팀 태그를 매깁니다. "
         "설명이나 스키마 설명 없이, 아래 예시와 똑같은 형태의 JSON 객체 하나만 실제 값으로 채워서 출력하세요.\n\n"
         "예시: {{\"topics\": [\"기술\", \"경쟁사\"], \"teams\": [\"프로덕트\", \"마케팅\"]}}\n\n"
         "{format_instructions}"),
        ("human", "{article}"),
    ]).partial(format_instructions=p2.get_format_instructions())
    | llm | p2
)

# 3개 체인 동시 실행
monitor_v1 = RunnableParallel(
    analysis=chain_analysis,
    brief=chain_brief,
    tags=chain_tags,
)

r = monitor_v1.invoke({"article": SAMPLE})
print(f"📌 {r['analysis'].headline} | 영향도 {r['analysis'].impact_score}")
print(f"🏷  {r['tags'].topics} / 팀: {r['tags'].teams}")

In [12]:
# 대응 라우팅 — RunnableLambda + Pydantic
class PRAction(BaseModel):
    priority: Literal["alert_now", "daily_brief", "archive"]
    channel: Literal["slack_urgent", "slack_daily", "notion"]
    message: str

def route_pr(x) -> PRAction:
    a = x["analysis"]
    if a.sentiment == "negative" and a.impact_score >= 7:
        return PRAction(priority="alert_now", channel="slack_urgent",
                       message=f"🚨 즉시 대응: {a.headline}")
    if a.impact_score >= 4:
        return PRAction(priority="daily_brief", channel="slack_daily",
                       message=f"📋 브리핑: {a.headline}")
    return PRAction(priority="archive", channel="notion",
                   message=f"📁 보관: {a.headline}")

monitor_v2 = monitor_v1 | RunnableLambda(route_pr)
print(monitor_v2.invoke({"article": SAMPLE}))

priority='daily_brief' channel='slack_daily' message='📋 브리핑: NovaTech, B2B 챗봇 시장 1위'


In [13]:
# 여러 기사 batch — 매일 들어오는 수십 건
ARTICLES = [
    {"article": SAMPLE},
    {"article": "NovaTech 일부 고객사 데이터 유출 의혹. 회사는 부인. 보안 업계 우려 제기."},
    {"article": "정부, AI 산업 5조원 지원금 편성. 스타트업·연구기관 대상."},
]
for a in monitor_v2.batch(ARTICLES):
    icon = {"alert_now": "🚨", "daily_brief": "📋", "archive": "📁"}[a.priority]
    print(f"{icon} {a.message}")

📋 📋 브리핑: NovaTech, B2B 챗봇 시장 1위
🚨 🚨 즉시 대응: 고객사 데이터 유출 의혹
📋 📋 브리핑: 정부, AI 산업 지원


---
## 🎓 마무리

같은 3개 부품으로 PR 모니터링 시스템 완성. **도메인이 달라도 패턴은 동일**.

**확장**: SNS 카피 체인 / 영문 번역 체인 / 경쟁사 비교 체인 등 `RunnableParallel`에 한 줄씩만 추가.